# பாடம் 12 - பொறியாளர் ஸ்க்ராட்ச்பேட் உடன் உரையாடல் வரலாறு குறைப்பு

இந்த நோட்புக் மைக்ரோசாஃப்ட் ஏஜெண்ட் ஃப்ரேம்வொர்க் பயன்படுத்தி நீண்ட உரையாடல்களில் சூழலை எப்படி நிர்வகிப்பது என்பதை வெளிப்படுத்துகிறது. உரையாடல்கள் வளர்ந்தவுடன், டோக்கன் எண்ணிக்கை அதிகரிக்கிறது — இறுதியில் மாடலின் சூழல் ஜன்னல் பாளையை கடந்துபோகிறது. இதை நாம் **சூழல் சுருக்கல் மாதிரியோடு** மற்றும் **ஏஜெண்ட் ஸ்க்ராட்ச்பேடு** மூலம் நிரந்தர நினைவகத்துடன் சமாளிக்கின்றோம்.

## நீங்கள் கற்றுக் கொள்வது:
1. **எதற்கு சூழல் நிர்வகிப்பு அவசியம்**: டோக்கன் வரம்புகள் மற்றும் சூழல் ஜன்னல்களை புரிந்துகொள்வது
2. **சூழல்-அறிவுடைய ஏஜெண்ட்கள்**: தங்களது உரையாடல் சூழலை நிர்வகிக்கும் ஏஜெண்ட்கள் உருவாக்கல்
3. **சூழல் சுருக்கல் மாதிரி**: உரையாடல் வரலாரை சுருக்கவும் கருவிகள் பயன்படுத்தல்
4. **ஏஜெண்ட் ஸ்க்ராட்ச்பேடு**: சூழல் குறைப்புக்குப் பிறகும் நிலைத்திருக்கும் நினைவகம்

## முன் தேவைகள்:
- சூழல் மாறிலிகள் அமைக்கப்பட்ட Azure OpenAI அமைப்பு
- முந்தைய பாடங்களிலிருந்து அடிப்படையான ஏஜெண்ட் கருத்துக்களை புரிந்துகொள்வது


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## காரணம் சூழல் மேலாண்மை முக்கியம்

ஒவ்வொரு LLMக்கும் ஒரு పరిమితமான **சூழல் ஜன்னல்** உள்ளது — அது ஒரு கோரிக்கையில் செயலாக்கக்கூடிய максимум டோக்கன்களின் எண்ணிக்கை. பல-சுற்று உரையாடல் முன்னேறும்போது:

- **டோக்கன் எண்ணிக்கை நேரியல் முறையில் வளர்கிறது** ஒவ்வொரு பயனர் செய்தியும் உதவியாளர் பதிலும் சேர்க்கையில்.
- **உரையாசிரியர் டோக்கன்கள் செலவில் முன்னிலை வகிக்கின்றன** ஏனெனில் முழுமையான வரலாறு ஒவ்வொரு முறையும் மீண்டும் அனுப்பப்படும்.
- இறுதியில் உரையாடல் **சூழல் ஜன்னலை மீறுகிறது** மற்றும் மாதிரி சுருக்கம் செய்கிறது அல்லது தவறு நிகழ்கிறது.

### சூழல் மேலாண்மைக்கு உத்திகள்

| உத்தி | அது எப்படி வேலை செய்கிறது | நன்மை-பெருக்கம் |
|---|---|---|
| **சுருக்குதல்** | பழைய செய்திகளை நீக்குதல் | ஆரம்ப சூழலை இழக்கும் |
| **சுருக்கம்** | பழைய செய்திகளை ஒரு சுருக்கமாக கூட்டுதல் | சில விவரங்கள் இழக்கும், ஆனால் முக்கிய புள்ளிகள் பாதுகாக்கப்படும் |
| **Scratchpad / வெளிப்புற நினைவகம்** | உரையாடலுக்கு வெளியே முக்கிய தகவல்களை சுருக்கி வைத்தல் | கருவி அழைப்புகள் தேவைப்படும், ஆனால் எந்த குறைக்கும் நிலையும் கடக்கலாம் |

இந்த நோட்டுபுத்தகத்தில் நாம் **சுருக்கத்துடன்** **scratchpad கருவி** ஒன்றை ஒன்றிணைக்கிறோம், இதனால் முகவர் உரையாடல் வரலாறு சுருக்கப்பட்டாலும் தொடர்ச்சியை பராமரிக்க முடியும்.


## ஒரு சூழல் அறிவு முகவரியை உருவாக்குதல்


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## நீண்ட உரையாடலை சிமுலேட் செய்வது

சந்ததியில் உள்ள அணிகலன்களை எப்படி சேமிப்பது என்று பார்க்க ஒரு பல-அடுக்க உரையாடலை கழிக்கலாம். முகவர் முக்கிய விவரங்களை (விருப்பங்கள், பட்ஜெட், பயண தேதி) அடுத்த அடுக்குகளுக்கு இடையில் வைத்திருக்க வேண்டும் மற்றும் தொடர்ச்சியை காண்பிக்க வேண்டும்.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

எஜেন্ট் முன்னைய உரையாடல்கள் பற்றிய பின்னணி நினைவில் வைத்துக்கொள்வதை கவனியுங்கள் — அது ஜப்பான், சுஷி, கோயில்கள், புகைப்படக் கலை, $3000 பட்ஜெட், தனியாக பயணம் மற்றும் ஏப்ரல் மாதக்காலம் பற்றி அறிந்துள்ளது. ஒரு குறுகிய உரையாடலில் இது நன்றாக வேலை செய்யும், ஆனால் உரையாடல் நீளும் போது முழு வரலாறு மீண்டும் அனுப்புவது விலைமதிப்படுகிறது.

பின்னணி சேர்க்கையைப் பார்க்க மற்ற்திசைய உரையாடலை தொடர்வோம்:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

உரையாடல் வளர்ந்தால், கூடுதலாக கூடிய உள்ளடக்கத்தை சுருங்கிய வடிவமாக்க ஒரு **சுருக்குதல் கருவி** பயன்படுத்தலாம். முகவர் முக்கிய முன்னுரிமைகளை பதிவு செய்ய இந்த கருவியை அழைக்கிறார், எனவே பழைய செய்திகள் நீக்கப்பட்டாலும், முக்கியமான தகவல் பாதுகாக்கப்படுகிறது.

இந்த முறை நுட்பமான வரலாறு குறைப்புக்கான அடித்தளம்:
1. முகவர் உரையாடலிலிருந்து முக்கியவுள்ள விஷயங்களை கண்டுபிடிக்கிறார்
2. அவற்றை நிலைநிறுத்த சுருக்கும் கருவியை அழைக்கிறார்
3. பழைய செய்திகள் பாதுகாப்புடன் நீக்கலாம் ஏனெனில் சுருக்கம் முக்கியமானதை எடுத்துக்காட்டுகிறது

கீழே, முகவர் கற்றுக் கொண்டதை சுருக்கமாக பதிவு செய்யக்கூடிய `summarize_preferences` கருவியை வரையறுக்கிறோம்.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework ஐ பயன்படுத்தி நீண்ட நேரம் இயங்கி கொள்கின்ற முகவரி உரையாடல்களில் சூழலை எப்படி நிர்வகிக்க வேண்டும் என்று கற்றுக்கொண்டீர்கள்:

### முக்கியக் கருத்துக்கள்
- **சூழல் ஜன்னல்கள் எல்லைக்குட்பட்டவை** — உரையாடல் வரலாற்றிலுள்ள ஒவ்வொரு டோக்கனும் செலவு ஆகும் மற்றும் வரம்புக்குள் கணக்கிடப்படுகிறது.
- **சுருக்கமிடும் கருவிகள்** முகவரிக்கு சேர்க்கப்பட்ட சூழலை தொகுத்து சுருக்கமான சுருக்கங்களாக மாற்ற அனுமதிக்கின்றன, முக்கியத் தகவலை பாதுகாத்து டோக்கன் கோப்பை குறைக்கின்றன.
- **முகவர் ஸ்கிராட்ச்பேட்டுகள்** பேச்சு குறைப்பை எதிர்கொள்ளும் எந்த உரையாடலையும் தாண்டி நிலையான வெளிப்புற நினைவகத்தை வழங்குகின்றன.

### நீங்கள் உருவாக்கியது
- பலமுறை உரையாடல்களில் தொடர்ச்சியை பராமரிக்கும் **சூழல் அறிவூட்டப்பட்ட முகவர்**
- முக்கிய பயனர் விவரங்களை சுருக்கமான வடிவில் பதிவுசெய்யும் **சுருக்கும் கருவி** (`summarize_preferences`)
- சூழல் பாதுகாப்பும் மாற்றங்களை கையாள்வதும் காட்டும் **பலமுறை உரையாடல்**

### எதிர்கால பயன்பாடுகள்
- **வாடிக்கையாளர் சேவை பாட்டுகள்**: நீண்ட உதவி அமர்வுகளில் விருப்பங்களை நினைவில் வைத்திருக்க
- **பிரத்தியேக உதவியாளர்கள்**: சூழலை மீண்டும் விளக்காமல் தொடர்ச்சியான திட்டங்களை கண்காணிக்க
- **கல்வி வழிகாட்டிகள்**: பல தொடர்புகளுக்கு இடையில் மாணவர் முன்னேற்றத்தை பராமரிக்க

### அடுத்த படிகள்
- கோப்பில் அடிப்படையாய் நிலைநிறுத்தப்பட்ட முழு ஸ்கிராட்ச்பேட் கருவியை செயல்படுத்தவும்
- சுருக்கத்திற்குப் பிறகு தானாக வரலாறு குறைப்பை சேர்க்கவும்
- அர்த்த நினைவக தேடலுக்கு வெக்டர் தரவுத்தளங்களுடன் இணைக்கவும்
- முழு சூழல் உடன் திங்கட்கிழமைகளுக்குப் பிறகு உரையாடல்களை மீண்டும் தொடங்கக்கூடிய முகவர்களை உருவாக்கவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**விரோதக் குறிப்பு**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சித்தாலும், தானாக செய்யப்பட்ட மொழிபெயர்ப்புகளில் தவறுகள் அல்லது தவறான தகவல்கள் இருக்கக்கூடும் என்பதைத் தயவுசெய்து கவனத்தில் கொள்ளவும். அச்சுப்பதிவு செய்யப்பட்ட மொழியில் உள்ள முதன்மை ஆவணம் அதிகாரபூர்வமான மூலமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்படும் எந்தவொரு தவறோ அல்லது தவறான விளக்கமோ தொடர்பாக நாம் பொறுப்பேற்றுக் கொள்ளமுடியாது.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
